# 1. Import Libraries

In [ ]:
import os
import sys
import json
import numpy as np
import tensorflow as tf

# 2. File Paths

In [ ]:
scripts_path = os.path.abspath(os.path.join('..', 'Scripts'))
if scripts_path not in sys.path:
    sys.path.append(scripts_path)

data_dir = os.path.abspath(os.path.join('..', 'Data'))
save_dir = os.path.abspath(os.path.join('..', 'SavedModels'))

# 3. Load Models & Data

## 1. Data

In [ ]:
train_data = np.load(os.path.join(data_dir, 'train_data.npy')).astype(np.float32)
num_items = train_data.shape[1]

## 2. Models

In [ ]:
from model import Encoder, Decoder, VAE, RSVD

# 4. Training

## 1. VAE

### 1. Import Hyperparameters

In [ ]:
vae_progress_file = os.path.join(save_dir, 'tuning_progress_vae.json')
with open(vae_progress_file, 'r') as f:
    best_vae_params = json.load(f)['best_params']

print(f"\n[INFO] Melatih ulang VAE dengan parameter terbaik: {best_vae_params}")

### 2. Construct Model

In [ ]:
encoder = Encoder(hidden_dims=best_vae_params['hidden_dims'], latent_dim=best_vae_params['latent_dim'], dropout_rate=best_vae_params['dropout_rate'])
decoder = Decoder(hidden_dims=best_vae_params['hidden_dims'][::-1], output_dim=num_items)
final_vae = VAE(encoder, decoder)

_ = final_vae(train_data[:1]) 
optimizer = tf.keras.optimizers.Adam(learning_rate=best_vae_params['learning_rate'])
final_vae.compile(optimizer=optimizer)

# Fitur Early Stopping: Berhenti jika loss tidak membaik selama 10 epoch berturut-turut
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='loss', patience=10, restore_best_weights=True, verbose=1
)

### 3. Training Model

In [ ]:
history = final_vae.fit(
    train_data, train_data, 
    epochs=100, # Naik drastis dari 15
    batch_size=best_vae_params['batch_size'], 
    callbacks=[early_stopping],
    verbose=1 # Tampilkan progress bar agar kita tahu pergerakannya
)

#### 4. Save Weights
    Save model weights after finished training

In [ ]:
final_vae.save_weights(os.path.join(save_dir, 'best_vae_weights.weights.h5'))
print("[SUCCESS] Bobot VAE Final berhasil disimpan!")

#### 5. Latent Matrix Z
    Get latent matrix from VAE for training RSVD

In [ ]:
print("\n[INFO] Mengekstrak Matriks Laten Z Final...")
z_mean, _ = final_vae.encoder.predict(train_data, batch_size=best_vae_params['batch_size'])
latent_matrix_Z_final = z_mean

## 2. RSVD

### 1. Import Weights

In [ ]:
rsvd_progress_file = os.path.join(save_dir, 'tuning_progress_rsvd.json')
with open(rsvd_progress_file, 'r') as f:
    best_rsvd_params = json.load(f)['best_params']

print(f"\n[INFO] Melatih ulang RSVD dengan parameter terbaik: {best_rsvd_params}")

### 2. Construct Model

In [ ]:
final_rsvd = RSVD(
    n_factors=best_rsvd_params['n_factors'], 
    learning_rate=best_rsvd_params['learning_rate'], 
    lambda_reg=best_rsvd_params['lambda_reg'], 
    epochs=100 # Naik drastis agar dekomposisi sempurna
)

### 3. Training Model

In [ ]:
final_rsvd.fit(latent_matrix_Z_final)

### 4. Save Weights

In [ ]:
np.save(os.path.join(save_dir, 'best_U.npy'), final_rsvd.U)
np.save(os.path.join(save_dir, 'best_Sigma.npy'), final_rsvd.Sigma)
np.save(os.path.join(save_dir, 'best_V.npy'), final_rsvd.V)
print("[SUCCESS] Matriks RSVD Final berhasil disimpan!")